# prep.virus

Virus were detected in a previous work (McLeish2024) using a BLAST-based pipeline. Because I don't have access to the original files but only a few datafiles provided by F., we need to convert the data from the format in which they were provided to a format that is useful for me. The final output rows contain:

- library
- acronym
- taxid
- scientific_name

In [2]:
import pandas as pd
import requests
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

┌────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│      name      │                                                                                  description                                                                                   │
│    varchar     │                                                                                    varchar                                                                                     │
├────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ D_bacteriaHits │ This table contains all the MOTUS hits obtained, regardless of their status. It contains the library where the detection took place, taxid, scientific name and the PAB labels │
│ D_PABHits      │ T

## Assign NCBI taxon identifier to virus species

The file provided by Fernando requires some actions to turn it from its matrix form into a list of Library - Otu, which will be much easier to analyze later.

We also have a metadata file (S4) that will allow us to assign a taxid to each viral species. To obtain taxon ids, we will need to query the NCBI api. 

In [3]:
def extract_taxonomy_from_gene_records(host_name):
    url = "https://api.ncbi.nlm.nih.gov/datasets/v2"
    url = f'{url}/gene/accession/{host_name}/dataset_report'.replace(' ', '%20')
    r = requests.get(url)
    
    try: 
        r.raise_for_status()
    except requests.HTTPError:
        return pd.NA
    try:
        u = str(r.json()['reports'][0]['gene']['tax_id'])
    except KeyError:
        return pd.NA
    except IndexError:
        return pd.NA
    return u


Reading metadata (S4), which contains a few interesting attributes such as the NCBI accession that we will use to find taxids.

In [4]:
virus = pd.read_csv("input/mcleish24.TableS4.csv", sep=';')
virus['OTU_taxid'] = virus['NCBI_accession'].apply(extract_taxonomy_from_gene_records)
virus

,OTU name,Family,NCBI_title,OTU_reference,Genome,%_ID_mean,%_ID_min,%_ID_max,BLAST_matches,NCBI_accession,OTU_taxid
0,Alfamovirus 1,Bromoviridae,Alfalfa mosaic virus RNA 1,AMV1,(+)ssRNA,96.6,94.4,99.2,4,NC_001495,12321
1,Alfamovirus 1,Bromoviridae,Alfalfa mosaic virus RNA 2,AMV2,(+)ssRNA,94.1,94.1,94.1,2,NC_002024,12321
2,Alphaendornavirus 1,Endornaviridae,Cucumis melo endornavirus,CmEV,dsRNA,96.5,87.9,100.0,4174,NC_029064,1776177
3,Alphaendornavirus 2,Endornaviridae,Bell pepper endornavirus,BPEV,dsRNA,100.0,100.0,100.0,2,NC_015781,<NA>
4,Alphaendornavirus 3,Endornaviridae,Hordeum vulgare endornavirus,HvEV,dsRNA,96.6,87.2,100.0,4002,NC_028949,1774276
...,...,...,...,...,...,...,...,...,...,...,...
172,Comovirus,Secoviridae,Arabidopsis latent virus-1 RNA1,ALaV1,(+)ssRNA,99.5,97.3,100.0,52,MH899120,<NA>
173,Comovirus,Secoviridae,Arabidopsis latent virus-1 RNA2,ALaV2,(+)ssRNA,97.6,94.0,100.0,30,MH899121,<NA>
174,Varicosavirus 1,Rhabdoviridae,Lettuce big-vein associated virus RNA 1,LBVaV1,(-)ssRNA,95.3,93.3,97.3,12,NC_011558,1985698
175,Varicosavirus 1,Rhabdoviridae,Lettuce big-vein associated virus RNA 2,LBVaV2,(-)ssRNA,94.9,90.6,98.0,32,NC_011568,1985698


## Load processed Virus hits

Now we load the original data. As it is formated as a matrix, we will need to melt it into a 1-row-1-detection dataframe.

In [5]:
hits = pd.read_csv("input/val_158_otu_host_df.csv", sep=';')
hits = hits.melt(
    id_vars=['Library', 'Host', 'Host_family', 'Habitat', 'Site', 'Collection'],
    value_vars=hits.columns[6:], 
).rename(columns={'variable': "OTU", 'value': "hit"}).query('hit == 1').copy()
hits['Collection'] = hits['Collection'].apply(lambda x: x.split("_")[0])
hits[['Library', 'Host', 'Host_family', 'Habitat', 'Site', 'Collection', 'OTU']]


,Library,Host,Host_family,Habitat,Site,Collection,OTU
74,PV091,Galium verum,Rubiaceae,Oak,Q4,Q4P,AEV1
332,PV059,Tragopogon sp,Asteraceae,Wasteland,E4,E4P,AEYV
425,PV159,Carduus bourgeanus,Asteraceae,Edge,L1,L1V,AEYV
448,PV182,Lactuca serriola,Asteraceae,Edge,L3,L3V,AEYV
537,PV498,Rubia peregrina,Rubiaceae,Crop,H3,H3P,AEYV
...,...,...,...,...,...,...,...
44458,PV172,Picris echioides,Asteraceae,Edge,L2,L2V,YSV
44487,PV201,Convolvulus arvensis,Convolvulaceae,Edge,L1,L1F,YSV
44625,PV047,Zea mays,Poaceae,Crop,Z2,Z2V,ZYMV
44638,PV061,Cucumis melo,Cucurbitaceae,Crop,M1,M5V,ZYMV


## Rename hits

To merge S4 and this file, we need to use a handcrafted dictionary (`mapping-otus-curated.csv`).

In [6]:
rename_map = pd.read_csv("input/mapping-otus-curated.csv", sep="\t", index_col=0)[['otu', 'hit']].set_index('otu').to_dict()['hit']
hits['OTU_rn'] = hits['OTU'].apply(lambda x: rename_map[x])
hits[hits['OTU_rn'].isna()][['OTU']].value_counts()

Series([], Name: count, dtype: int64)

## Final merge

Finally, we merge the two files. 

In [7]:
virus_hits = pd.merge(hits, virus, left_on='OTU_rn', right_on='OTU_reference', how='left')[['Library', 'OTU_reference', 'OTU_taxid', 'NCBI_title']]
virus_hits = virus_hits.rename(
    columns={'Library': 'library', 'OTU_reference': 'acronym', 'OTU_taxid': 'taxid', 'NCBI_title': 'scientific_name'}
).sort_values(by=['library', 'acronym'])
virus_hits

,library,acronym,taxid,scientific_name
300,PV001,CMV3,12305,Cucumber mosaic virus RNA 3
286,PV001,CmEV,1776177,Cucumis melo endornavirus
529,PV001,GMMV3,578305,Gayfeather mild mottle virus RNA 3
937,PV001,RuCMV,378833,Rubus chlorotic mottle virus
301,PV002,CMV3,12305,Cucumber mosaic virus RNA 3
...,...,...,...,...
515,PV589,CYDV_RPV,2170100,Cereal yellow dwarf virus - RPV
1575,PV589,WYDV_GPV,572239,Wheat yellow dwarf virus - GPV
488,PV590,CMV_satRNA,<NA>,Cucumber mosaic virus satellite RNA
1047,PV590,RuCMV,378833,Rubus chlorotic mottle virus


## Save

In [8]:
virus_hits.to_csv("output/hits.virus.csv", sep=";", index=None)

In [9]:
db.save_dataframe(
    virus_hits, table_name="D_virusHits", 
    description="This table contains all the virus hits. It contains the library where the detection took place, taxid, scientific name and the virus acronym"
)

Saved D_virusHits to db.2025-10-27


In [10]:
db.conn.close()